In [25]:
import numpy as np

class MyMLPClassifier:
    def __init__(self, hidden_layer_sizes=(100,), activation='relu', alpha=0.0001,
                 learning_rate_init=0.001, max_iter=200, random_state=None, verbose=False):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.activation = activation
        self.alpha = alpha
        self.learning_rate_init = learning_rate_init
        self.max_iter = max_iter
        self.random_state = random_state
        self.verbose = verbose
        self.weights = []
        self.biases = []

    def _initialize_weights_and_biases(self, input_size, output_size):
        #initializez legaturile dintre neuroni si biaseurile
        #layer_sizes va contine numarul de neuroni din fiecare layer
        layer_sizes = [input_size] + list(self.hidden_layer_sizes) + [output_size]
        for i in range(1, len(layer_sizes)):
            weight = np.random.randn(layer_sizes[i - 1], layer_sizes[i]) * np.sqrt(2. / layer_sizes[i - 1])
            bias = np.zeros((1, layer_sizes[i]))
            self.weights.append(weight)
            self.biases.append(bias)

    #functia de activare aplicata dupa fiecare strat
    def _activation_function(self, z):
        if self.activation == 'relu':
            return np.maximum(0, z)
        elif self.activation == 'tanh':
            return np.tanh(z)
        elif self.activation == 'sigmoid':
            return 1 / (1 + np.exp(-z))
        else:
            raise ValueError("Invalid activation function. Supported activations: 'relu', 'tanh', 'sigmoid'.")

    #derivata folosita pentru invatare
    def _derivative_activation_function(self, z):
        if self.activation == 'relu':
            return (z > 0).astype(float)
        elif self.activation == 'tanh':
            return 1 - z ** 2
        elif self.activation == 'sigmoid':
            return np.exp(-z) / (1 + np.exp(-z)) ** 2
        else:
            raise ValueError("Invalid activation function. Supported activations: 'relu', 'tanh', 'sigmoid'.")

    def fit(self, X, y):
        self._initialize_weights_and_biases(X.shape[1], 1)
        X = np.array(X)
        y = np.reshape(np.array(y), (-1, 1))

        for i in range(self.max_iter):
            #forward propagation
            a = [X] #va stoca activarile fiecarui strat
            #parcurgem straturile
            for weight, bias in zip(self.weights[:-1], self.biases[:-1]):
                #produsul scalar dintre activarile stratului anterior si greutatilor dintre stratul anterior si cel curent + bias 
                z = np.dot(a[-1], weight) + bias
                a.append(self._activation_function(z))

            # output layer with sigmoid (binary classification)
            z = np.dot(a[-1], self.weights[-1]) + self.biases[-1]
            a_out = 1 / (1 + np.exp(-z))  # sigmoid
            a.append(a_out)

            #loss(eroarea patratica medie)
            if self.verbose and i % 10 == 0:
                loss = np.mean(np.square(y - a[-1]))
                print(f"Epoch {i}, Loss: {loss}")
                
            # backward propagation
            delta = a_out - y #eroarea de la output
            #listele care vor contine derivatele pt actualizare
            grads_w = []
            grads_b = []

            #parcurgem straturile invers
            for l in reversed(range(len(self.weights))):
                a_prev = a[l]
                dw = np.dot(a_prev.T, delta) / X.shape[0] #derivata fata de weights
                db = np.mean(delta, axis=0, keepdims=True) #derivata fata d bias

                grads_w.insert(0, dw)
                grads_b.insert(0, db)

                #ne intoarcem la stratul anterior
                if l > 0:
                    delta = np.dot(delta, self.weights[l].T) * self._derivative_activation_function(a[l])

            # update weights(invatarea propriu zisa)
            for l in range(len(self.weights)):
                self.weights[l] -= self.learning_rate_init * grads_w[l]
                self.biases[l] -= self.learning_rate_init * grads_b[l]

    

            

    def predict(self, X):
        #forward propagation
        a = X
        for weight, bias in zip(self.weights[:-1], self.biases[:-1]):
            z = np.dot(a, weight) + bias
            a = self._activation_function(z)
        # output layer with sigmoid (binary classification)
        output = 1 / (1 + np.exp(-np.dot(a, self.weights[-1]) - self.biases[-1]))
        return (output >= 0.5).astype(int)


In [26]:
#Transforma fiecare text intr-un vector care spune de cate ori apare fiecare cuvant.
def bag_of_words(train_inputs, test_inputs):
    from sklearn.feature_extraction.text import CountVectorizer
    vectorizer = CountVectorizer()
    
    train_features = vectorizer.fit_transform(train_inputs)
    test_features = vectorizer.transform(test_inputs)

    train_features = train_features.toarray()
    test_features = test_features.toarray()

    return train_features, test_features

#spune cat de important e un cuvant intr-un text.
def tf_idf(train_inputs, test_inputs):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(max_features=50)

    train_features = vectorizer.fit_transform(train_inputs)
    test_features = vectorizer.transform(test_inputs)

    train_features = train_features.toarray()
    test_features = test_features.toarray()

    return train_features, test_features

import gensim
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

#model inspirat din word2vec
#Transforma fiecare propozitie intr-un vector numeric de dimensiune vector_size
#Invata automat contextul si semnificatia cuvintelor (nu doar pozitia lor ca in BoW/TF-IDF)
def extractFeaturesDoc2Vec(trainInputs, testInputs, vector_size=100, epochs=20):
    # Crearea setului de date etichetat (TaggedDocument)(rupe textul in cuvinte si il eticheteaza)
    train_data = [TaggedDocument(words=text.split(), tags=[str(i)]) for i, text in enumerate(trainInputs)]
    test_data = [text.split() for text in testInputs]  # Doar cuvintele, fără etichete
    
    # Antrenarea modelului Doc2Vec
    model = Doc2Vec(vector_size=vector_size, window=5, min_count=1, workers=4, epochs=epochs)
    model.build_vocab(train_data)
    model.train(train_data, total_examples=model.corpus_count, epochs=model.epochs)
    
    # Extrage vectorii pentru fiecare document
    #Pentru fiecare text din train/test, infer_vector il transforma intr-un vector de dimensiune 100
    trainFeatures = [model.infer_vector(text.split()) for text in trainInputs]
    testFeatures = [model.infer_vector(text) for text in test_data]

    trainFeatures = np.array(trainFeatures)
    testFeatures = np.array(testFeatures)
    
    return trainFeatures, testFeatures


In [27]:
def splitData(inputs,outputs):
    np.random.seed(7)

    no_samples = len(inputs)
    indexes = [i for i in range(no_samples)]
    train_sample = np.random.choice(indexes, int(0.8 * no_samples), replace=False)
    test_sample = [i for i in indexes if i not in train_sample]

    train_inputs = [inputs[i] for i in train_sample]
    train_outputs = [outputs[i] for i in train_sample]
    test_inputs = [inputs[i] for i in test_sample]
    test_outputs = [outputs[i] for i in test_sample]

    return train_inputs, train_outputs, test_inputs, test_outputs

In [28]:
import pandas as pd
def data_reader(filename):
    df = pd.read_csv(filename)
    numeric_cols = df.select_dtypes(include='number').columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

    subset = df[['Text', 'Sentiment']]

    text = [subset.iat[i, 0] for i in range(len(subset))]
    sentiment = [subset.iat[i, 1] for i in range(len(subset))]
    labels = list(set(sentiment))

    return text, sentiment, labels

In [29]:
from sklearn.neural_network import MLPClassifier
import numpy as np

def predictTool(trainFeatures, testFeatures, labels, classes):
    ann = MLPClassifier(hidden_layer_sizes=(100, 50),    
    activation='relu',              
    solver='adam',                     
    learning_rate='adaptive',         
    learning_rate_init=0.001,        
    max_iter=700,                    
    early_stopping=True,                  
    random_state=42,                
    verbose=True    )
    ann.fit(trainFeatures, labels)

    # Prezicere
    predictions = ann.predict(testFeatures)

    return predictions.tolist()

In [30]:
def predictMyANN(trainFeatures, testFeatures, labels, classes):
    mlp = MyMLPClassifier(hidden_layer_sizes=(100,50), activation='relu', max_iter=1000, learning_rate_init=0.005,verbose=True)
    mlp.fit(trainFeatures, labels)
    prediction=mlp.predict(testFeatures)
    TestOutputs = ['negative' if elem == 0 else 'positive' for elem in prediction]
    return TestOutputs

In [31]:

import gensim 

# Load Google's pre-trained Word2Vec 

modelPath = 'GoogleNews-vectors-negative300.bin'

word2vecModel300 = gensim.models.KeyedVectors.load_word2vec_format(modelPath, binary=True,limit=100000) 
print(word2vecModel300.most_similar('support'))
print("vec for house: ", word2vecModel300["house"])

[('supporting', 0.6251285076141357), ('Support', 0.6044272780418396), ('supported', 0.6009396314620972), ('backing', 0.6007589101791382), ('supports', 0.5269277691841125), ('assistance', 0.5207138061523438), ('supportive', 0.5110024809837341), ('encouragement', 0.5081833600997925), ('Supporting', 0.4793948531150818), ('wholehearted', 0.46596306562423706)]
vec for house:  [ 1.57226562e-01 -7.08007812e-02  5.39550781e-02 -1.89208984e-02
  9.17968750e-02  2.55126953e-02  7.37304688e-02 -5.68847656e-02
  1.79687500e-01  9.27734375e-02  9.03320312e-02 -4.12109375e-01
 -8.30078125e-02 -1.45507812e-01 -2.37304688e-01 -3.68652344e-02
  8.74023438e-02 -2.77099609e-02  1.13677979e-03  8.30078125e-02
  3.57421875e-01 -2.61718750e-01  7.47070312e-02 -8.10546875e-02
 -2.35595703e-02 -1.61132812e-01 -4.78515625e-02  1.85546875e-01
 -3.97949219e-02 -1.58203125e-01 -4.37011719e-02 -1.11328125e-01
 -1.05957031e-01  9.86328125e-02 -8.34960938e-02 -1.27929688e-01
 -1.39648438e-01 -1.86523438e-01 -5.71289

In [32]:
def featureComputation(model, data):
    features = []
    phrases = [ phrase.split() for phrase in data]
    for phrase in phrases:
        # compute the embeddings of all the words from a phrase (words of more than 2 characters) known by the model
        vectors = [model[word] for word in phrase if (len(word) > 2) and (word in model.index_to_key)]
        if len(vectors) == 0:
            result = [0.0] * model.vector_size
        else:
            result = np.sum(vectors, axis=0) / len(vectors)
        features.append(result)
    return features


In [37]:
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


print('\nEmotions')
fp = 'reviews_mixed.csv'
text, sentiment, labels = data_reader(fp)
trainInputs, trainOutputs, testInputs, testOutputs = splitData(text, sentiment)
#trainFeatures, testFeatures =bag_of_words(trainInputs, testInputs)
trainFeatures, testFeatures = tf_idf(trainInputs, testInputs)
#trainFeatures, testFeatures = extractFeaturesDoc2Vec(trainInputs, testInputs)
# trainFeatures = featureComputation(word2vecModel300, trainInputs)
# testFeatures = featureComputation(word2vecModel300, testInputs)

scaler = StandardScaler()
trainFeatures = scaler.fit_transform(trainFeatures)
testFeatures = scaler.transform(testFeatures)

labels2= [0 if elem == 'negative' else 1 for elem in trainOutputs]
print('tool')
computedOutputs = predictTool(trainFeatures, testFeatures, trainOutputs, len(set(labels)))

print('MyANN')
myComputedOutputs = predictMyANN(np.array(trainFeatures), np.array(testFeatures), labels2, len(set(labels)))

inverseTestOutputs = ['negative' if elem == 'positive' else 'positive' for elem in testOutputs]

accuracyByTool = accuracy_score(testOutputs, computedOutputs)
accuracyByToolInverse = accuracy_score(inverseTestOutputs, computedOutputs)
print('Accuracy score by tool:', max(accuracyByTool, accuracyByToolInverse))

accuracyByMe = accuracy_score(testOutputs, myComputedOutputs)
accuracyByMeInverse = accuracy_score(inverseTestOutputs, myComputedOutputs)
print('Accuracy score by me:', max(accuracyByMe, accuracyByMeInverse))
print('\n')

print('Output computed by tool:  ', computedOutputs)
print('\n')
print('Output computed by me:    ', myComputedOutputs)
print('\n')
print('Real output:              ', testOutputs)


Emotions
tool
Iteration 1, loss = 0.88304430
Validation score: 0.294118
Iteration 2, loss = 0.74485324
Validation score: 0.352941
Iteration 3, loss = 0.63927265
Validation score: 0.294118
Iteration 4, loss = 0.55818676
Validation score: 0.352941
Iteration 5, loss = 0.49461130
Validation score: 0.588235
Iteration 6, loss = 0.44453469
Validation score: 0.647059
Iteration 7, loss = 0.40302483
Validation score: 0.705882
Iteration 8, loss = 0.36753270
Validation score: 0.705882
Iteration 9, loss = 0.33638542
Validation score: 0.705882
Iteration 10, loss = 0.30844847
Validation score: 0.705882
Iteration 11, loss = 0.28362037
Validation score: 0.705882
Iteration 12, loss = 0.26137289
Validation score: 0.705882
Iteration 13, loss = 0.24141604
Validation score: 0.705882
Iteration 14, loss = 0.22371918
Validation score: 0.705882
Iteration 15, loss = 0.20774088
Validation score: 0.705882
Iteration 16, loss = 0.19314976
Validation score: 0.705882
Iteration 17, loss = 0.17988871
Validation score: 

In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
text_input = ["By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement."]

# Preprocesare
tfidfVectorizer=TfidfVectorizer()
trainFeatures = tfidfVectorizer.fit_transform(trainInputs).toarray()

text_input_features = tfidfVectorizer.transform(text_input).toarray()

scaler = StandardScaler()
trainFeatures_scaled = scaler.fit_transform(trainFeatures)
text_input_features=scaler.transform(text_input_features)

# Predictii folosind modelele
predicted_tool = predictTool(trainFeatures, text_input_features, trainOutputs, len(set(labels)))
predicted_myann = predictMyANN(trainFeatures, text_input_features, labels2, len(set(labels)))

print('\nSentiment estimat:')
print('Cu Tool: ',predicted_tool)
print('Cu MyANN: ',predicted_myann)


# AZURE

from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

key = "EnR4AULaXGwWtetzCvtPEFSmPncN9TT6grph3FNk2PHfzLjugfswJQQJ99BDAC5RqLJXJ3w3AAAaACOGnRJA"
endpoint = "https://analizate.cognitiveservices.azure.com/"

credential = AzureKeyCredential(key)
client = TextAnalyticsClient(endpoint=endpoint, credential=credential)

documents = [text_input[0]]
response = client.analyze_sentiment(documents=documents)[0]

print("Sentiment AZURE:", response.sentiment)

Iteration 1, loss = 0.67889369
Validation score: 0.705882
Iteration 2, loss = 0.67034164
Validation score: 0.705882
Iteration 3, loss = 0.66216381
Validation score: 0.705882
Iteration 4, loss = 0.65429964
Validation score: 0.705882
Iteration 5, loss = 0.64670227
Validation score: 0.705882
Iteration 6, loss = 0.63934783
Validation score: 0.705882
Iteration 7, loss = 0.63212474
Validation score: 0.705882
Iteration 8, loss = 0.62504285
Validation score: 0.705882
Iteration 9, loss = 0.61806493
Validation score: 0.705882
Iteration 10, loss = 0.61112144
Validation score: 0.705882
Iteration 11, loss = 0.60414996
Validation score: 0.705882
Iteration 12, loss = 0.59712950
Validation score: 0.705882
Validation score did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.
Epoch 0, Loss: 0.252497993396001
Epoch 10, Loss: 0.25086167392115044
Epoch 20, Loss: 0.24932554572969748
Epoch 30, Loss: 0.24783936119082942
Epoch 40, Loss: 0.2463930010396166
Epoch 50, Loss: 0.24500563864091